In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS gold;

-- Drop if exists to ensure a clean schema start
DROP TABLE IF EXISTS gold.fact_sales;

CREATE TABLE gold.fact_sales (
    sales_sk STRING,      -- The MD5 Hash from Silver
    customer_sk BIGINT,   -- The Surrogate Key from Dim_Customers
    store_id INT,
    amount DECIMAL(10,2),
    event_date DATE,
    load_timestamp TIMESTAMP
) USING DELTA
PARTITIONED BY (event_date);

In [0]:
from pyspark.sql import functions as F

# 1. Load Silver Tables
fact_df = spark.table("silver.fact_transactions")
dim_df = spark.table("silver.dim_customers")

# 2. Perform the Surrogate Key Replacement with a LEFT JOIN
gold_df = (fact_df.alias("f")
    .join(F.broadcast(dim_df).alias("d"), 
        (F.col("f.customer_id") == F.col("d.customer_id")) & 
        (F.col("f.event_time") >= F.col("d.start_date")) & 
        (F.col("f.event_time") <= F.col("d.end_date")), 
        "left") # <--- Changed to LEFT
    .select(
        F.col("f.transaction_sk").alias("sales_sk"),
        # Optional: Coalesce NULLs to a '-1' (Late Arriving Dimension handling)
        F.coalesce(F.col("d.customer_sk"), F.lit(-1)).alias("customer_sk"),
        F.col("f.store_id"),
        F.col("f.amount"),
        F.col("f.event_date"),
        F.current_timestamp().alias("load_timestamp")
    ))

# 3. Write to Gold
(gold_df.write
    .mode("overwrite")
    .saveAsTable("gold.fact_sales"))

In [0]:
%sql
select * from gold.fact_sales

In [0]:
%sql
select count(*) from gold.fact_sales